# 🧹 Fase 3: Data Preparation & Feature Engineering
**Geo-Price Analyzer** — Cleaning, Encoding, dan Split Data

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
plt.style.use('seaborn-v0_8-darkgrid')

def format_rupiah(value, _=None):
    if value >= 1e9: return f'Rp {value/1e9:.1f}M'
    elif value >= 1e6: return f'Rp {value/1e6:.0f}Jt'
    else: return f'Rp {value:,.0f}'

df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'raw', 'jabodetabek_house_price.csv'))
print(f'📂 Raw data: {df.shape[0]} baris × {df.shape[1]} kolom')

## 3.1 Seleksi Kolom yang Relevan

In [ ]:
# Pilih kolom yang relevan untuk modeling
cols_keep = ['price_in_rp', 'city', 'bedrooms', 'bathrooms', 'land_size_m2',
             'building_size_m2', 'carports', 'garages', 'floors']
df = df[cols_keep].copy()
print(f'📋 Kolom dipilih: {list(df.columns)}')
df.head()

## 3.2 Handling Missing Values

In [ ]:
print('=== SEBELUM ===')
print(df.isnull().sum())
print(f'Total baris: {len(df)}')

In [ ]:
# Drop baris yang tidak punya harga (target)
df = df.dropna(subset=['price_in_rp'])

# Imputasi fitur numerik dengan median
num_cols = ['bedrooms','bathrooms','land_size_m2','building_size_m2','carports','garages','floors']
for col in num_cols:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        n_miss = df[col].isnull().sum()
        df[col] = df[col].fillna(med)
        print(f'   ✅ {col}: {n_miss} missing → median ({med})')

# Drop baris tanpa kota
df = df.dropna(subset=['city'])

print(f'\n✅ Sisa missing: {df.isnull().sum().sum()}')
print(f'Total baris: {len(df)}')

## 3.3 Penghapusan Outliers (IQR)

$$\text{Outlier jika } x < Q1 - 1.5 \cdot IQR \text{ atau } x > Q3 + 1.5 \cdot IQR$$

In [ ]:
Q1 = df['price_in_rp'].quantile(0.25)
Q3 = df['price_in_rp'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
n_before = len(df)

mask = (df['price_in_rp'] >= lower) & (df['price_in_rp'] <= upper)

# Juga filter luas tanah/bangunan yang tidak masuk akal
mask &= (df['land_size_m2'] > 0) & (df['land_size_m2'] < 5000)
mask &= (df['building_size_m2'] > 0) & (df['building_size_m2'] < 5000)

df = df[mask].reset_index(drop=True)
n_removed = n_before - len(df)

print(f'IQR Harga: Q1={format_rupiah(Q1)}, Q3={format_rupiah(Q3)}')
print(f'Upper bound: {format_rupiah(upper)}')
print(f'Dihapus: {n_removed} baris ({n_removed/n_before*100:.1f}%)')
print(f'Tersisa: {len(df)} baris')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['price_in_rp'], bins=40, color='#2ecc71', edgecolor='white')
axes[0].set_title('Distribusi Harga (Setelah Cleaning)', fontweight='bold')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
axes[1].boxplot(df['price_in_rp'])
axes[1].set_title('Boxplot Harga (Setelah Cleaning)', fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(format_rupiah))
plt.tight_layout()
plt.show()

## 3.4 Feature Engineering

In [ ]:
df['rasio_tanah_bangunan'] = df['land_size_m2'] / df['building_size_m2'].replace(0, np.nan)
df['rasio_tanah_bangunan'] = df['rasio_tanah_bangunan'].fillna(1.0)
df['total_ruangan'] = df['bedrooms'] + df['bathrooms']

print('✅ Fitur baru:')
print('   + rasio_tanah_bangunan (luas tanah / luas bangunan)')
print('   + total_ruangan (KT + KM)')
df[['land_size_m2','building_size_m2','rasio_tanah_bangunan','total_ruangan']].describe().round(2)

## 3.5 Encoding Lokasi

In [ ]:
le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])

print('📋 Label Encoding:')
for i, c in enumerate(le.classes_):
    count = (df['city'] == c).sum()
    print(f'   {c:30s} → {i} ({count} listing)')

## 3.6 Train-Test Split (80:20)

In [ ]:
feature_cols = ['land_size_m2','building_size_m2','bedrooms','bathrooms',
                'carports','garages','floors','city_encoded',
                'rasio_tanah_bangunan','total_ruangan']

X = df[feature_cols]
y = df['price_in_rp']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} sampel × {X_train.shape[1]} fitur')
print(f'Test:  {X_test.shape[0]} sampel × {X_test.shape[1]} fitur')
print(f'\nNaN check — X_train: {X_train.isnull().sum().sum()} | X_test: {X_test.isnull().sum().sum()}')

## 3.7 Simpan Data Processed

In [ ]:
save_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'jabodetabek_processed.csv')
df.to_csv(save_path, index=False)
print(f'✅ Disimpan ke: {save_path}')
print(f'   {df.shape[0]} baris × {df.shape[1]} kolom')

---
## 📝 Ringkasan
| Step | Hasil |
|---|---|
| Seleksi kolom | 9 kolom relevan dipilih |
| Missing values | Imputasi median |
| Outliers | IQR + filter luas anomali |
| Feature Engineering | +rasio_tanah_bangunan, +total_ruangan |
| Encoding | Label Encoding pada `city` |
| Split | 80:20 train-test |

**Selanjutnya →** `03_Modeling_Evaluation.ipynb`